In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for custom src imports later
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

substitution distribution, per query mean substitution rate, zero vs any substitutions, substitution count breakdown,

substitution matrix

mean entropy, entrpy variance, frac. conserved, entropy (no gaps),

number of homologs per query, evolutionary persistence (species breadth),

rg metrics

In [ ]:
import pandas as pd
df_combined = pd.read_pickle("/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/combined_blast_results.pkl")

In [ ]:
# Basic shape / diagnostic
print(f"Total rows: {len(df_combined):,}")
print(f"Unique queries: {df_combined['UniqueID'].nunique()}")
print(f"Unique (query, motif) pairs: {df_combined.groupby(['UniqueID', 'orig_motif_index']).ngroups}")
print(f"\nGroup breakdown:")
print(df_combined["group"].value_counts())

print(f"\nHomologs per query-motif pair:")
per_motif = df_combined.groupby(['UniqueID', 'orig_motif_index']).size()
print(per_motif.describe())

print(f"\nSelf-hits (qseq == hseq):")
print((df_combined["qseq_rg_region"] == df_combined["hseq_rg_region"]).value_counts())

print(f"\nExample of orig_motif_index meaning:")
print(df_combined[["UniqueID", "orig_motif_index", "motif_start", "motif_end", "qseq_rg_region"]].head(10))

print(f"\nSpecies diversity per query-motif:")
species_per_motif = df_combined.groupby(['UniqueID', 'orig_motif_index'])['species'].nunique()
print(species_per_motif.describe())

In [ ]:
# Are self-hits same species as the query, or conserved identical across species?
self_hits = df_combined[df_combined["qseq_rg_region"] == df_combined["hseq_rg_region"]]

# Check species distribution of self-hits
print("Self-hits species distribution (top 10):")
print(self_hits["species"].value_counts().head(10))

print(f"\nTotal unique species in self-hits: {self_hits['species'].nunique()}")
print(f"Total unique species in all hits: {df_combined['species'].nunique()}")

In [ ]:
df_combined

In [ ]:
import src.analysis_visualization.homolog_recruitment as hr

results = hr.run_phase1_homolog_analyses(df_combined, dataset="homologs")

# Access individual pieces
stats_df = results["stats_df"]

In [ ]:
import src.analysis_visualization.homolog_entropy as he

# Need harmonized labels + motif_uid first — use the df from Phase 1 if loaded
# or harmonize fresh here:
import src.analysis_visualization.homolog_recruitment as hr
df = hr.harmonize_group_labels(df_combined)
df = hr.add_motif_uid(df)

results = he.run_entropy_analysis(
    df,
    min_homologs=10,           # optional: skip motifs with fewer homologs
    include_query_in_alignment=True,
    dataset="homologs",
)

# Access individual objects
features_df = results["features_df"]
test_results = results["test_results"]

In [ ]:
df_combined

In [ ]:
import src.analysis_visualization.homolog_entropy as he
import src.analysis_visualization.homolog_recruitment as hr

# Harmonize & add motif_uid
df = hr.harmonize_group_labels(df_combined)
df = hr.add_motif_uid(df)

# Run position-level analysis
pos_results = he.run_position_level_analysis(
    df,
    min_homologs=10,
    min_positions_per_motif=2,
    dataset="homologs",
)

per_pos = pos_results["per_position_df"]              # ~10k positions
per_aa_results = pos_results["per_aa_results"]        # 20-row test table
comp_df = pos_results["within_motif_comp_df"]         # per-motif RG-vs-non-RG

In [ ]:
#### compare this to a neutral background

### some random notes

could i generalize this and find within the IDR landscape the genetic patterns?
use this to find regions within the IDR landscape of humans?
can i apply this to repeats as well?